# Deferral-Aware Digital Twin — full demo

Run the cells in order, top to bottom. Steps 2 and 3 are the slow ones (about 20 minutes together);
everything after that is quick.

**Before you start, have these four files ready on this computer:**

| File | What it is |
|---|---|
| `scaffold_v7.zip` | the project code |
| `phase2_answers.json` | your saved answers from the Twin Interview page |
| a voice clip | 30–60 seconds of you speaking, any format |
| a photo | front-facing, whole face, mouth closed, good light |

**Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Check the GPU and install the main libraries  *(about 3 minutes)*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install "gradio>=5,<6" faiss-cpu sentence-transformers pyyaml pandas \
    accelerate bitsandbytes "transformers>=4.57,<5"
print('done')

## 2. Install the voice and avatar models  *(about 18 minutes — start it and leave it)*

Three separate environments, because these models pin conflicting library versions.
Run this cell once per Colab session.

In [ ]:
!pip -q install coqui-tts
import os; os.environ['COQUI_TOS_AGREED'] = '1'
print('voice ok — now the avatar, this is the slow part')

In [ ]:
# Upload the project zip when the file picker appears (any scaffold_*.zip)
import os, glob; os.chdir('/content')
from google.colab import files
up = list(files.upload().keys())
z = next((f for f in up if f.endswith('.zip')), None)
assert z, f'No .zip among the uploaded files: {up}'
!rm -rf /content/digital-twin && mkdir -p /content/digital-twin
!unzip -q -o "$z" -d /content/digital-twin
print('unpacked', z); !ls /content/digital-twin/src | head

In [ ]:
!bash /content/digital-twin/src/setup_avatar.sh      # SadTalker env + GFPGAN weights
!bash /content/digital-twin/src/setup_wav2lip.sh     # Wav2Lip lip-sync (the one that worked)
!cp /content/digital-twin/src/enhance_frames.py /content/enh.py
print('avatar ready')

## 3. Connect Google Drive  *(so a disconnect costs you nothing)*

Your answers, voice and photo are copied here. If Colab resets, step 5 restores them automatically and you skip the uploads.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/twin_assets'
!mkdir -p "$DRIVE" && ls "$DRIVE"

## 4. Upload your three files

Pick all three at once: `phase2_answers.json`, your voice clip, your photo.

*Skip this cell if you already did it once and Drive has them.*

In [ ]:
import os; os.chdir('/content')
from google.colab import files
up = list(files.upload().keys())
ANSWERS = next((f for f in up if f.endswith('.json')), None)
VOICE   = next((f for f in up if f.lower().endswith(('.wav','.m4a','.mp3','.aac','.ogg','.mp4'))), None)
PHOTO   = next((f for f in up if f.lower().endswith(('.jpg','.jpeg','.png','.heic'))), None)
print('answers:', ANSWERS, '| voice:', VOICE, '| photo:', PHOTO)

## 5. Build the twin  *(under a minute)*

This does everything on its own: splits your answers into what the twin may use and what it is tested on, builds the fact index, computes your behaviour profile, converts the audio, fixes the photo rotation, wires up voice and avatar, and backs it all up to Drive.

In [ ]:
import sys; sys.path.insert(0, '/content/digital-twin/src')
import os; os.chdir('/content/digital-twin')
from build_twin import build

twin = build(answers=globals().get('ANSWERS'),
             voice=globals().get('VOICE'),
             photo=globals().get('PHOTO'),
             work_dir='/content', drive=DRIVE)
print()
print(twin.profile.narrative()[:600])

## 6. Load the language model  *(about 3 minutes)*

Phi-3-mini in 4-bit, which leaves room on the T4 for the voice and avatar models.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'microsoft/Phi-3-mini-4k-instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map='auto', trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.bfloat16))
model.eval()
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'left'

@torch.inference_mode()
def generate_batch(system_prompts, questions, max_new_tokens=90):
    texts = [tok.apply_chat_template(
                [{'role':'system','content':s}, {'role':'user','content':q}],
                tokenize=False, add_generation_prompt=True)
             for s, q in zip(system_prompts, questions)]
    enc = tok(texts, return_tensors='pt', padding=True, truncation=True,
              max_length=3000).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
            for o in out]

llm_fn = lambda sp, q, ctx: generate_batch([sp], [q])[0]
twin.llm_fn = llm_fn
print(generate_batch(['You are helpful. Answer in one short sentence.'], ['Say hello.']))

## 7. Launch the demo

Open the **public URL** that prints below — that is the link to show on the projector.

**What to demonstrate, in this order:**
1. Press *Pick a held-out question* — one the twin has never seen. Its answer appears beside your real one.
2. Ask something your interview never covered → **🔴 Deferred**, not an invented answer. This is the point of the project.
3. Ask a new situation it can reason about → **🟡 Inferred**, and open the *Behaviour profile* tab to show what it reasoned from.
4. *Teach the twin*: correct an answer, ask again, it has changed.
5. *Tell it about you*: type a few new facts, then ask about one — it answers from them immediately.
6. Tick **Speak** and **Avatar video** to hear and see it answer as you.

Corrections and added facts are written to disk, so they survive a restart.

In [ ]:
demo = twin.app()
demo.launch(share=True)

## 8. Optional: the Phase 2 numbers for your report

Scores the twin on every held-out question and against your own consistency ceiling — the best score any twin of you could get.

In [ ]:
from eval_phase2 import main as eval_main
summary = eval_main('data/benchmark/phase2', 'phase2_me', llm_batch_fn=generate_batch)

---
### If the runtime disconnects

Re-run steps 1, 2, 3, then **skip step 4** and run step 5 with no uploads — it restores your answers, voice and photo from Drive. Then steps 6 and 7. The corrections you taught the twin are restored too.